# 03 — QHE Benchmarks

**A³E-DNS Thesis Artifact · Chapter 6 · §6.5, §6.14**

This notebook exercises the Ring-LWE engine at the thesis parameters
(§6.5: `n = 256`, `q = 4096`) and compares the NTT-based multiplication
to a schoolbook reference. It is a **correctness** notebook, not a
security notebook. The IND-CPA claim is a citation of the published
Ring-LWE result (§5.3.5).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from src.models.qhe_engine import QHEConfig, QHEEngine

cfg = QHEConfig(n=256, q=4096)
engine = QHEEngine(cfg)
engine.keygen()

msg = np.array([1, 0, 1, 1, 0, 0, 1, 0] + [0] * (256 - 8), dtype=np.int64)
ct = engine.encrypt(msg)
recovered = engine.decrypt(ct)

print('round-trip ok:', bool(np.array_equal(recovered, msg)))

### NTT vs schoolbook

On the thesis parameters `n = 256`, `q = 4096`, the NTT multiplication agrees with the schoolbook negacyclic convolution in the ring `Z_q[x] / (x^n + 1)`.

In [ ]:
import time
from src.models.qhe_ntt import NTTContext, ntt_multiply, negacyclic_multiply

rng = np.random.default_rng(20261130)
a = rng.integers(0, 4096, size=256, dtype=np.int64)
b = rng.integers(0, 4096, size=256, dtype=np.int64)
ctx = NTTContext(n=256, q=4096)

t0 = time.perf_counter()
out_ntt = ntt_multiply(a, b, n=256, q=4096, context=ctx)
t_ntt = time.perf_counter() - t0

t0 = time.perf_counter()
out_ref = negacyclic_multiply(a, b, q=4096)
t_ref = time.perf_counter() - t0

print('NTT agrees with schoolbook:', bool(np.array_equal(out_ntt % 4096, out_ref % 4096)))
print(f'NTT: {t_ntt*1000:.2f} ms')
print(f'schoolbook: {t_ref*1000:.2f} ms')

### A100 emulator

The A100 emulator returns the reported throughput trends (not absolute values). See `docs/HARDWARE_EMULATION.md` for the exact scope of the emulation claim.

In [ ]:
from src.hardware_simulation.a100_emulation import A100Emulator

emu = A100Emulator()
for op in ['ntt_256', 'ntt_1024', 'homomorphic_add', 'homomorphic_mul']:
    print(f'{op:20s} {emu.estimate_latency_ms(op):>8.2f} ms')